# 07 · NEWS2 sobre la cohorte TRIAGE — el punto de operación a batir

**Pregunta que responde:** ¿cuál es el punto de operación de NEWS2 —tasa de alertas *y*
sensibilidad al mismo tiempo— sobre una cohorte donde el desenlace es real? Es lo que el
notebook 06 **no pudo** dar, y es exactamente lo que `SCRUM-58` necesita para comparar.

| | |
|---|---|
| **Decisión que lo habilita** | **ADR-009** — segunda cohorte |
| **Notebook previo** | `06_baseline_news2.ipynb` (la tasa de referencia sobre MIMIC) |
| **Entrada** | `data/processed/triage-zenodo/triage.parquet` |
| **Dataset** | TRIAGE (`zenodo.org/records/4963759`), **CC0**, 1.303 pacientes de guardia. Sin registro ni DUA. |
| **Salida** | `data/interim/curva_operacion_news2.parquet` y figuras en `data/interim/figuras/` |
| **Semilla** | ninguna: nada se sortea acá. |

> ### Por qué hizo falta una segunda cohorte
> Sobre el demo de MIMIC-IV-ED se puede medir **cuánto alerta** NEWS2 pero no **cuánto
> acierta**: el horizonte de 24 h de la etiqueta v2 no discrimina —estadía mediana 5,8 h—
> y la prevalencia colapsa al 66,8 %. Sin sensibilidad no existe la "comparativa a
> sensibilidad igualada", que es la tesis del proyecto. Acá el desenlace es mortalidad a
> 30 días real, con **4,1 %** de prevalencia.

> ### Lo que esta cohorte NO puede hacer
> Una sola fila por paciente: **no sirve para la LSTM** (`SCRUM-57`, sigue dependiendo de
> MIMIC) ni para la etiqueta v2. El desenlace de acá es a 30 días y de ingreso a UCI, y
> **no se mezcla** con la etiqueta de MIMIC en una misma métrica. Ver ADR-009.

In [ ]:
import sys
from pathlib import Path

RAIZ = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(RAIZ))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.metrics import roc_auc_score

from src.data import alertas, triage
from src.viz import estilo

estilo.aplicar()
pd.set_option("display.width", 120)

INTERIM = RAIZ / "data" / "interim"
FIGURAS = INTERIM / "figuras"
FIGURAS.mkdir(parents=True, exist_ok=True)

df = pd.read_parquet(RAIZ / "data" / "processed" / "triage-zenodo" / "triage.parquet")
p = df[df.puntuable].copy()
p["score"] = p["news2_score"].astype(int)
p["rojo"] = p["news2_rojo_aislado"].astype(bool)
p["alerta_sistema"] = p["news2_riesgo"].isin(alertas.RIESGOS_QUE_ALERTAN)

print(f"pacientes: {len(df)}   puntuables: {len(p)}")
print(f"muerte a 30 días: {int(p.muerte_30d.sum())} ({p.muerte_30d.mean():.1%})")
print(f"ingreso a UCI   : {int(p.ingreso_uci.sum())} ({p.ingreso_uci.mean():.1%})")

## 1 · La prevalencia, que es lo que cambia todo

**Qué espero encontrar:** una prevalencia de un dígito, dentro de la banda real del
problema (0,22 %–5 %), y no el 66,8 % del demo de MIMIC.

Es la diferencia que justifica el ADR-009. Una prevalencia del 66,8 % no describe un
problema de detección temprana: describe una cohorte donde casi todos terminan internados.
Con 4,1 % sí hay algo que detectar.

In [ ]:
comparacion = pd.DataFrame([
    {"cohorte": "MIMIC-IV-ED demo", "n": 553, "unidad": "tomas",
     "desenlace": "etiqueta v2 (t+1h, t+24h]", "prevalencia": "66.8 %",
     "¿sensibilidad medible?": "NO"},
    {"cohorte": "TRIAGE", "n": len(p), "unidad": "pacientes",
     "desenlace": "muerte 30 d", "prevalencia": f"{p.muerte_30d.mean():.1%}",
     "¿sensibilidad medible?": "sí"},
    {"cohorte": "TRIAGE", "n": len(p), "unidad": "pacientes",
     "desenlace": "ingreso a UCI", "prevalencia": f"{p.ingreso_uci.mean():.1%}",
     "¿sensibilidad medible?": "sí"},
]).set_index("cohorte")
print(comparacion.to_string())

## 2 · Cuánto acierta NEWS2 acá

**Qué espero encontrar:** una sensibilidad mediocre —del orden del 50 %— con una tasa de
alertas alta. Es lo que reporta la literatura para NEWS en guardia, y es el supuesto que
justifica el proyecto entero: la escala **grita mucho y se pierde la mitad**.

Se aplica la **regla real del sistema** (`riesgo != Bajo`), la misma del notebook 06 y la
misma del trigger de la base. No `score >= 5`.

In [ ]:
def operacion(mascara, evento):
    y = p[evento].to_numpy().astype(bool)
    a = np.asarray(mascara, dtype=bool)
    return {
        "tasa de alertas": a.mean(),
        "sensibilidad": a[y].mean() if y.sum() else np.nan,
        "especificidad": (~a[~y]).mean(),
        "alertas": int(a.sum()),
        "eventos captados": int(a[y].sum()),
        "eventos totales": int(y.sum()),
    }

filas = []
for evento, nombre in [("muerte_30d", "muerte a 30 días"), ("ingreso_uci", "ingreso a UCI")]:
    r = operacion(p["alerta_sistema"], evento)
    filas.append({"desenlace": nombre,
                  "tasa de alertas": f"{r['tasa de alertas']:.1%}",
                  "sensibilidad": f"{r['sensibilidad']:.1%}",
                  "especificidad": f"{r['especificidad']:.1%}",
                  "captados": f"{r['eventos captados']} de {r['eventos totales']}"})

print(pd.DataFrame(filas).set_index("desenlace").to_string())
print()
print(f"AUROC del score NEWS2 — muerte 30 d: {roc_auc_score(p.muerte_30d, p.score):.3f}")
print(f"AUROC del score NEWS2 — UCI        : {roc_auc_score(p.ingreso_uci, p.score):.3f}")

**Lo que se ve:** NEWS2 alerta en una de cada cuatro admisiones y **se pierde casi la mitad
de las muertes**. El AUROC de 0,73 para mortalidad es consistente con lo publicado para
NEWS en guardia — no es que el motor esté mal, es que la escala rinde así en este entorno.

Eso es precisamente el hueco que el proyecto quiere llenar, ahora con un número propio en
vez de una cita.

## 3 · La curva de operación: lo que `SCRUM-58` tiene que batir

**Qué espero encontrar:** que al barrer el umbral del score se dibuje la frontera de
NEWS2, y que el punto del sistema caiga **sobre** ella o cerca.

Acá está el entregable del notebook. La comparativa "a sensibilidad igualada" necesita
saber, para cada nivel de sensibilidad, **cuántas alertas cuesta**. Cualquier modelo futuro
se compara contra esta curva: a la misma sensibilidad, ¿alerta menos?

In [ ]:
def barrido(evento):
    y = p[evento].to_numpy().astype(bool)
    filas = []
    for umbral in range(0, int(p.score.max()) + 2):
        a = (p.score >= umbral).to_numpy()
        filas.append({"umbral": umbral,
                      "tasa_alertas": a.mean(),
                      "sensibilidad": a[y].mean() if y.sum() else np.nan,
                      "especificidad": (~a[~y]).mean()})
    return pd.DataFrame(filas)

curva = barrido("muerte_30d")
print(curva.assign(
    tasa_alertas=lambda d: (100 * d.tasa_alertas).round(1),
    sensibilidad=lambda d: (100 * d.sensibilidad).round(1),
    especificidad=lambda d: (100 * d.especificidad).round(1),
).to_string(index=False))

sistema = operacion(p["alerta_sistema"], "muerte_30d")
print()
print(f"punto REAL del sistema (riesgo != Bajo): "
      f"tasa {sistema['tasa de alertas']:.1%}, sensibilidad {sistema['sensibilidad']:.1%}")

In [ ]:
fig, ax = plt.subplots(figsize=(8.6, 4.4))

ax.plot(100 * curva.tasa_alertas, 100 * curva.sensibilidad,
        color=estilo.TERRACOTA, linewidth=2, marker="o", markersize=5,
        markerfacecolor=estilo.SUPERFICIE, markeredgewidth=1.6, zorder=3)

# Offsets a mano: con el desplazamiento uniforme, las etiquetas de los umbrales 3 y 4
# y la del punto del sistema se pisaban entre si.
OFFSETS = {3: (10, 4, "left"), 4: (-8, 12, "right"), 5: (9, -4, "left"), 7: (9, -4, "left")}
for _, fila in curva.iterrows():
    if int(fila.umbral) in OFFSETS:
        dx, dy, ha = OFFSETS[int(fila.umbral)]
        ax.annotate(f"score >= {int(fila.umbral)}",
                    xy=(100 * fila.tasa_alertas, 100 * fila.sensibilidad),
                    xytext=(dx, dy), textcoords="offset points",
                    ha=ha, fontsize=8.5, color=estilo.GRIS)

ax.scatter([100 * sistema["tasa de alertas"]], [100 * sistema["sensibilidad"]],
           s=150, marker="D", color=estilo.OLIVA, zorder=4,
           edgecolor=estilo.SUPERFICIE, linewidth=1.5)
ax.annotate("regla desplegada\n(riesgo != Bajo)",
            xy=(100 * sistema["tasa de alertas"], 100 * sistema["sensibilidad"]),
            xytext=(16, -26), textcoords="offset points",
            fontsize=9, color=estilo.OLIVA, linespacing=1.4,
            arrowprops=dict(arrowstyle="-", color=estilo.OLIVA, linewidth=1,
                            shrinkA=0, shrinkB=6))

ax.set_xlabel("Tasa de alertas (% de admisiones)")
ax.set_ylabel("Sensibilidad para muerte a 30 días (%)")
ax.set_title("La frontera de NEWS2: cualquier modelo tiene que quedar arriba y a la izquierda",
             loc="left", color=estilo.TINTA)
ax.set_xlim(-3, 103)
ax.set_ylim(-3, 103)
fig.savefig(FIGURAS / "07_curva_operacion_news2.png")
plt.show()

**Un resultado que corta para los dos lados, y por eso es interesante.**

*Para mortalidad a 30 días*, la regla desplegada queda **por dentro** de la curva del score
puro: con `score >= 4` se consigue **22,6 % de alertas con 59,3 % de sensibilidad**, mientras
que la regla real da **25,7 % con 53,7 %** — más alertas y menos aciertos.

*Para ingreso a UCI*, es al revés: la regla desplegada alcanza el 47,3 % de sensibilidad con
25,7 % de alertas, y el umbral de score más barato que iguala esa sensibilidad cuesta
**36,6 %**. Ahí el rojo aislado **gana, y por bastante**.

**La lectura, que es la que va a la tesis:** el rojo aislado captura deterioro fisiológico
agudo que exige atención *ahora* —una frecuencia respiratoria de 6 es una emergencia
inmediata— y eso se parece mucho más a "este paciente termina en UCI" que a "este paciente
muere dentro de 30 días". El resultado es coherente con la razón clínica por la que NEWS2
incluye esa regla, no contra ella.

Conclusión operativa: **no hay evidencia para sacar el rojo aislado**, y sí la hay para
elegir el desenlace con cuidado. Juzgar la regla sólo por mortalidad a 30 días sería
juzgarla por algo que no se propone predecir.

## 4 · El objetivo concreto para `SCRUM-58`

Con la curva se puede fijar la vara sin ambigüedad: **a la sensibilidad del sistema
desplegado (53,7 % para muerte a 30 días), ¿cuántas alertas necesita NEWS2?** Cualquier
modelo que quede por debajo de esa tasa, a esa misma sensibilidad, gana.

In [ ]:
objetivos = []
for evento, nombre in [("muerte_30d", "muerte a 30 días"), ("ingreso_uci", "ingreso a UCI")]:
    c = barrido(evento)
    s_sistema = operacion(p["alerta_sistema"], evento)["sensibilidad"]
    # el umbral de score más barato que alcanza al menos esa sensibilidad
    alcanzan = c[c.sensibilidad >= s_sistema]
    mejor = alcanzan.loc[alcanzan.tasa_alertas.idxmin()] if len(alcanzan) else None
    objetivos.append({
        "desenlace": nombre,
        "sensibilidad objetivo": f"{s_sistema:.1%}",
        "tasa de la regla desplegada": f"{operacion(p['alerta_sistema'], evento)['tasa de alertas']:.1%}",
        "tasa del mejor umbral de score": f"{mejor.tasa_alertas:.1%}" if mejor is not None else "—",
        "umbral": int(mejor.umbral) if mejor is not None else None,
    })

objetivo = pd.DataFrame(objetivos).set_index("desenlace")
print(objetivo.to_string())
print()
print("LA VARA: el modelo de SCRUM-56 tiene que alcanzar esa sensibilidad con una tasa")
print("de alertas MENOR que la del mejor umbral de score, que es el rival más exigente")
print("(no la regla desplegada, que es peor).")

In [ ]:
# Salida para SCRUM-58: la curva completa de los dos desenlaces.
salida = pd.concat([barrido(e).assign(desenlace=e) for e in ("muerte_30d", "ingreso_uci")],
                   ignore_index=True)
salida["cohorte"] = "triage-zenodo"
salida.to_parquet(INTERIM / "curva_operacion_news2.parquet", index=False)
print(f"guardado: {INTERIM / 'curva_operacion_news2.parquet'}  ({len(salida)} filas)")

## Conclusiones

**El punto de operación de NEWS2, para citar en la tesis:**

> Sobre la cohorte TRIAGE (1.300 pacientes de guardia puntuables), aplicando la regla que el
> sistema despliega, NEWS2 alerta en el **25,7 % de las admisiones** y detecta el **53,7 %
> de las muertes a 30 días** (29 de 54) y el **47,3 % de los ingresos a UCI** (80 de 169).
> AUROC 0,728 y 0,650 respectivamente.

**Qué se concluyó**

1. **La tesis por fin es medible.** Con prevalencia real del 4,1 % hay sensibilidad que
   medir, y por lo tanto existe la comparativa a sensibilidad igualada.
2. **NEWS2 grita mucho y acierta la mitad.** Una de cada cuatro admisiones alerta y se
   pierde casi la mitad de las muertes. Es el hueco que el proyecto quiere llenar, ahora
   con número propio en vez de cita.
3. **El rojo aislado corta para los dos lados, y eso lo valida.** Para mortalidad a 30
   días la regla desplegada queda por dentro de la curva del score (`score >= 4` da menos
   alertas y más sensibilidad); para ingreso a UCI **gana con claridad** (25,7 % de alertas
   contra 36,6 % del mejor umbral, a igual sensibilidad). Es coherente con la razón clínica
   por la que NEWS2 incluye esa regla: detecta deterioro que exige atención *ahora*, que se
   parece a "termina en UCI" y no a "muere en 30 días".
4. **La vara de `SCRUM-58` queda fijada** y guardada en
   `curva_operacion_news2.parquet`. El rival a batir es el **mejor umbral de score**, no la
   regla desplegada, que es más débil.

**Qué queda abierto**

- **`SCRUM-56`** — baseline XGBoost sobre este snapshot. Falta instalar `xgboost`.
- **`SCRUM-58`** — la comparativa, contra la curva de acá.
- **`SCRUM-57`** — la LSTM **sigue bloqueada**: esta cohorte no tiene series temporales.
  Depende de la base completa de MIMIC (`SCRUM-48`).
- Reescribir el **ADR-002**, que hablaba de "un dataset principal" cuando ahora hay dos
  cohortes con roles distintos.